In [ ]:
from ultralytics import RTDETR
from multiprocessing import freeze_support
import os

if __name__ == '__main__':
    freeze_support() 
    
    # RT-DETR 세그멘테이션 모델 로드 - 모델 경로 수정
    model = RTDETR('C:/Users/AI-LHJ/Desktop/YOLOv11/rtdetr-l.pt')
    
    # 세그멘테이션 학습
    results = model.train(
        data='C:/Users/AI-LHJ/Desktop/Wire2/data.yaml',
        epochs=500,
        batch=8,
        device='cuda',
        workers=8,
        optimizer='Adam',
        lr0=0.001,
        imgsz=640,
        patience=30,
        task='segment'  # 세그멘테이션 태스크 지정
    )

    # 학습된 모델 저장
    model_save_path = 'C:/Users/AI-LHJ/Desktop/Wire/RT-DETR_model.pt'
    model.save(model_save_path)
    print(f"Trained model saved to {model_save_path}")

    # 학습된 모델 테스트 
    test_image_path = 'C:/Users/AI-LHJ/Desktop/Wire/RT-DETR_image.jpg'
    if os.path.exists(test_image_path):
        results = model(test_image_path)
        print("Test prediction completed.")
        results.show()  # 세그멘테이션 결과 시각화

In [ ]:
from ultralytics import RTDETR
from multiprocessing import freeze_support
import os
import yaml

if __name__ == '__main__':
    freeze_support()
    
    # 데이터셋 설정 확인
    with open('C:/Users/AI-LHJ/Desktop/Wire2/data.yaml', 'r') as f:
        data_config = yaml.safe_load(f)
        print("Dataset configuration:", data_config)
    
    # RT-DETR 모델 로드
    model = RTDETR('C:/Users/AI-LHJ/Desktop/YOLOv11/rtdetr-l.pt')
    
    # 학습 설정
    results = model.train(
        data='C:/Users/AI-LHJ/Desktop/Wire2/data.yaml',
        epochs=500,
        batch=8,
        device='cuda',
        workers=8,
        optimizer='AdamW',          # AdamW optimizer 사용
        lr0=0.0005,                # 낮은 learning rate
        momentum=0.937,            
        warmup_epochs=5,           # warmup 적용
        patience=50,               # Early stopping patience 증가
        task='segment',            # segmentation task
        single_cls=False,          # 다중 클래스 활성화
        overlap_mask=True,         # mask overlap 허용
        mask_ratio=4,              # mask 해상도 비율
        retina_masks=True,         # high-quality mask 생성
        save=True,                 # 모델 저장
        save_period=10,            # 10 에폭마다 저장
        cache=True,                # 캐시 사용
        exist_ok=True,             # 기존 실험 덮어쓰기
        pretrained=True,           # 사전학습 가중치 사용
        verbose=True,              # 상세 로그 출력
        seed=42                    # 재현성을 위한 시드 설정
    )

    # 학습된 모델 저장
    model_save_path = 'C:/Users/AI-LHJ/Desktop/Wire/RT-DETR_seg_model.pt'
    model.save(model_save_path)
    print(f"Trained segmentation model saved to {model_save_path}")

    # 모델 검증
    metrics = model.val()
    print("Validation metrics:", metrics)

    # 테스트 이미지에 대한 예측
    test_image_path = 'C:/Users/AI-LHJ/Desktop/Wire/RT-DETR_image.jpg'
    if os.path.exists(test_image_path):
        results = model.predict(
            source=test_image_path,
            save=True,                # 결과 저장
            save_txt=True,            # 텍스트 형식으로 결과 저장
            save_conf=True,           # confidence score 저장
            save_crop=True,           # 크롭된 예측 저장
            show=True,                # 결과 시각화
            retina_masks=True,        # high-quality mask 생성
            line_width=2,             # 선 두께
            vid_stride=1              # 비디오 처리시 프레임 간격
        )
        
        # 결과 시각화 및 저장
        for r in results:
            im_array = r.plot()
            print(f"Segmentation completed. Results saved.")

    print("Training and evaluation completed.")